In [ ]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score

In [ ]:
file_path = "/content/MULTI-SENSOR ANOMALY & CONDITION.xlsx"
df = pd.read_excel(file_path)

In [ ]:
features = [
    'Temperature',
    'Vibration',
    'Pressure',
    'EnergyConsumption',
    'ProductionUnits'
]

X = df[features]


In [ ]:
X = X.fillna(X.mean())


In [ ]:
iso = IsolationForest(
    n_estimators=200,
    contamination=0.02,
    random_state=42
)

In [ ]:
iso.fit(X)

IsolationForest(contamination=0.02, n_estimators=200, random_state=42)

In [ ]:
df['anomaly_label'] = iso.predict(X)

In [ ]:
df['anomaly_score'] = iso.decision_function(X)

In [ ]:
anomalies = df[df['anomaly_label'] == -1]

In [ ]:
anomalies = anomalies.sort_values(by='anomaly_score')

In [ ]:
y_pred = iso.predict(X)

y_pred = (y_pred == -1).astype(int)

anomaly_percentage = (y_pred == 1).mean() * 100
print("Anomaly %:", anomaly_percentage)


Anomaly %: 2.0


In [ ]:
top_200 = anomalies[['Timestamp', 'MachineID'] + features].head(200)

top_200.to_excel("Top_200_Anomalies.xlsx", index=False)

print(top_200)

                Timestamp  MachineID  Temperature  Vibration   Pressure  \
4686  2025-07-15 06:00:00        134    49.666905   4.565849  43.980748   
17351 2026-12-24 23:00:00        119    47.830536   6.939574  19.825743   
1924  2025-03-22 04:00:00        145    48.081012   1.222529  21.650807   
11415 2026-04-21 15:00:00        148    42.276925   1.485924  20.541072   
18438 2027-02-08 06:00:00        115    43.997003   7.251840  45.370878   
...                   ...        ...          ...        ...        ...   
18098 2027-01-25 02:00:00        147    79.391731   1.855642  17.859315   
11665 2026-05-02 01:00:00        107    77.815892   5.322342  21.825035   
15879 2026-10-24 15:00:00        113    47.709749   3.474558  16.868036   
10294 2026-03-05 22:00:00        110    61.542095   2.268708  40.112423   
9872  2026-02-16 08:00:00        119    93.706062   7.032540  33.541271   

       EnergyConsumption  ProductionUnits  
4686          369.667775               61  
17351      

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [ ]:
X = df[features]
y = df['MaintenanceFlag']
X = X.fillna(X.mean())


In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)


In [ ]:
rf.fit(X, y)

RandomForestClassifier(n_estimators=200, random_state=42)

In [ ]:
y_pred = rf.predict(X)

In [ ]:
accuracy = accuracy_score(y, y_pred)
print("Accuracy:", accuracy)

Accuracy: 1.0


In [ ]:
df['anomaly_label'] = y_pred

In [ ]:
df['anomaly_score'] = rf.predict_proba(X)[:, 1]

In [ ]:
anomalies = df[df['anomaly_label'] == 1]

# Sort by score
anomalies = anomalies.sort_values(by='anomaly_score', ascending=False)

In [ ]:
anomaly_percentage = (y_pred == 1).mean() * 100
print("Anomaly %:", anomaly_percentage)

Anomaly %: 9.865


In [ ]:
top_200 = anomalies[['Timestamp', 'MachineID'] + features].head(200)

In [ ]:
top_200.to_excel("Top_200_Anomalies_RF.xlsx", index=False)

print(top_200)

                Timestamp  MachineID  Temperature  Vibration   Pressure  \
7522  2025-11-10 10:00:00        142    54.350974   6.632335  32.400392   
17984 2027-01-20 08:00:00        117    92.235328   6.349500  50.071032   
11728 2026-05-04 16:00:00        122    75.329717   2.804637  24.648434   
19815 2027-04-06 15:00:00        111    69.603338   6.293243  29.052761   
19685 2027-04-01 05:00:00        113    76.116361   7.297127  21.223929   
...                   ...        ...          ...        ...        ...   
4910  2025-07-24 14:00:00        144    72.652118   6.161090  33.369154   
179   2025-01-08 11:00:00        101    61.439910   2.844957  33.689592   
5702  2025-08-26 14:00:00        106    64.745144   4.952884  29.372905   
19587 2027-03-28 03:00:00        119    73.166720   2.565406  32.915951   
11561 2026-04-27 17:00:00        103    75.589168   6.170113  35.631782   

       EnergyConsumption  ProductionUnits  
7522          271.298148               98  
17984      

Deployment

In [ ]:
import pickle

In [ ]:
with open('isolation_forest_model.pkl', 'wb') as f:
    pickle.dump(iso, f)
print("Model saved successfully!")

Model saved successfully!


In [ ]:
with open('isolation_forest_model.pkl', 'rb') as f:
    model = pickle.load(f)

print("Model loaded successfully!")

Model loaded successfully!


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import pickle
with open('isolation_forest_model.pkl', 'rb') as f:
    model = pickle.load(f)
st.title("Anomaly Detection App")
st.write("Enter sensor values:")
temperature = st.number_input("Temperature")
vibration = st.number_input("Vibration")
pressure = st.number_input("Pressure")
energy = st.number_input("Energy Consumption")
production = st.number_input("Production Units")

if st.button("Predict"):
    data = pd.DataFrame([[temperature, vibration, pressure, energy, production]],
                        columns=['Temperature', 'Vibration', 'Pressure', 'EnergyConsumption', 'ProductionUnits'])

    prediction = model.predict(data)

    if prediction[0] == -1:
        st.error("⚠️ Anomaly Detected!")
    else:
        st.success("✅ Normal Condition")


Writing app.py


In [ ]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 103.5 MB/s eta 0:00:00


In [ ]:
!pip install pyngrok

In [ ]:
from pyngrok import ngrok
!ngrok config add-authtoken 3BCUlZOYCmltY7c9S45WXZuPXUh_7F339hb2LsrrfAt3vJ5cD

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://unproscriptive-dogmatically-laurie.ngrok-free.dev" -> "http://localhost:8501"
